# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 5: *Feature Interaction Analysis*
##### Version Number: 2.0
---
### Contents  
> 1. *Filter Dates for modeling*
> 2. *Split and Scale Data*
> 3. *Export Data*
---
### Notes
---
### Inputs
- `X`,`y`,`details` - Modeling sets
---
### Outputs 
- `X_scaled`,`y_reduced`,`details_reduced` - Scaled Modeling sets (sometimes with reduced modeling sets to preserve processor)
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions
from src.data_utils import create_interactions

# A space saving function to rank interactions
from src.data_utils import rank_variables_by_correlation

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling prep
from sklearn.preprocessing import MinMaxScaler

---

###  Load Data

In [ ]:
y = pd.read_csv("../data/processed/y.csv")
X = pd.read_csv("../data/processed/X.csv")
details = pd.read_csv("../data/processed/details.csv")

### (OPTIONAL) Downsize classes to save processor
- All classes are downsized for working on models

In [ ]:
final = pd.concat([details_full, X, y], axis=1)

In [ ]:
final['Date'] = pd.to_datetime(final['Date'])

# Number of rows to keep from classes 
n_rows = 10000

# Split by class
class0 = final[final['Target'] == 0]
class1 = final[final['Target'] == 1]
class2 = final[final['Target'] == 2]

# Sample classes down to specified rows
class0_sampled = class0.sample(n=n_rows, random_state=14)
class1_sampled = class1.sample(n=n_rows, random_state=14)
class2_sampled = class2.sample(n=n_rows, random_state=14)

# Combine classes back together
reduced = pd.concat([class0_sampled, class1_sampled , class2_sampled], ignore_index=True)

In [7]:
reduced['Target'].value_counts()

Target
0    10000
1    10000
2    10000
Name: count, dtype: int64

## Interaction Analysis

In [8]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target', 'grid_id', 'centroid_easting', 'centroid_northing',
       'geometry', 'fire_count', 'total_fire_damage','acres',  'dominant_section_description',
                'dominant_province_description','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','area_in_cali']

y_reduced = reduced['Target']
X_reduced = reduced.drop(columns=text_columns)
details_reduced = reduced[text_columns]

## Scaling

Scale all datasets and save back to dataframe format. MinMax Scaler used because majority of values are > 0

In [9]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_reduced)
X_scaled = pd.DataFrame(X_scaled, columns=X_reduced.columns, index=X_reduced.index)

### Rank variables by correlation strength

In [10]:
rank_variables_by_correlation(X_scaled,y_reduced)

,Feature,Correlation
0,2-Year Avg Fires,0.298295
1,Year,0.292077
2,Solar Radiation 7 Day Avg,0.276527
3,influence_zone,0.272304
4,Solar Radiation,0.255264
...,...,...
59,Standardized Precipitation Evapotranspiration ...,-0.029640
60,elevation_mean,-0.018936
61,Standardized Precipitation Index 180-Day,0.016513
62,Standardized Precipitation Index 30-Day,-0.001033


---

## 3. Export Data

In [12]:
X_reduced.to_csv('../data/processed/X_scaled.csv', index=False)
y_reduced.to_csv('../data/processed/y_reduced.csv', index=False)
details_reduced.to_csv('../data/processed/details_reduced.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
